In [16]:
import pandas as pd
import numpy as np

import pickle, os
os.makedirs('models', exist_ok=True)


import matplotlib.pyplot as plt
import seaborn as sns

from config import DATA_PATH

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

In [17]:
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [18]:

df = pd.read_excel(DATA_PATH, sheet_name='Sheet1')

In [19]:
# train/test 분리 (2025년 8월 기준)
cutoff_year, cutoff_month = 2025, 8
is_test = (df['연도'] > cutoff_year) | ((df['연도'] == cutoff_year) & (df['월'] >= cutoff_month))

train_df = df[~is_test].reset_index(drop=True)
test_df = df[is_test].reset_index(drop=True)

print("train:", train_df.shape)
print("test:", test_df.shape)

# 원핫인코딩 (수업 방식과 똑같이 fit은 train만, transform은 둘 다)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(train_df[['시도']])   # 대응표는 train으로만 생성

# 인코딩 결과를 새 컬럼들로 만들어서 붙이기
train_encoded = encoder.transform(train_df[['시도']])
test_encoded = encoder.transform(test_df[['시도']])

encoded_cols = encoder.get_feature_names_out(['시도'])

train_df[encoded_cols] = train_encoded
test_df[encoded_cols] = test_encoded

train_df
test_df

train: (731, 19)
test: (204, 19)


,시도,연도,월,총전력사용량(kWh),전년동월,세대수,냉방도일,난방도일,폭염일수(일),한파일수(일),...,시도_부산광역시,시도_서울특별시,시도_세종특별자치시,시도_울산광역시,시도_인천광역시,시도_전라남도,시도_전라북도,시도_제주특별자치도,시도_충청남도,시도_충청북도
0,강원도,2025,8,1434213477,1476779943,767034,8.2,0.0,11.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,강원도,2025,9,1384287487,1401551996,767540,3.4,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,강원도,2025,10,1193226667,1258740480,767989,0.0,5.9,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,강원도,2025,11,1315862219,1310014365,768570,0.0,16.4,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,강원도,2025,12,1464494147,1474912813,768900,0.0,22.4,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,충청북도,2026,3,2212648246,2369668082,802699,0.0,17.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
200,충청북도,2026,4,2161650683,2128868403,804351,0.0,11.6,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
201,충청북도,2026,5,2339858134,2017109073,806188,1.5,6.2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
202,충청북도,2026,6,2197221666,2116852957,807263,4.4,1.2,1.2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [20]:
exclude_cols = ['시도', '총전력사용량(kWh)',
                '주택용(%)', '일반용(%)', '교육용(%)', '산업용(%)', '농사용(%)', '가로등(%)', '심야(%)',
                '세대수']

X_train = train_df.drop(columns=exclude_cols)
y_train = train_df['총전력사용량(kWh)']
X_test = test_df.drop(columns=exclude_cols)
y_test = test_df['총전력사용량(kWh)']


In [21]:
#RandomForest
from sklearn.ensemble import RandomForestRegressor

#랜덤 포레스트 객체 생성
rf_model = RandomForestRegressor(
    n_estimators=100, # 트리를 몇개 만들지 지정 (많을수록 좋은건아니고 성능개선은 미미하나 계산시간만 늘어남)
    random_state=42,  # # 매번 실행해도 같은 결과가 나오게 고정하는 값
    n_jobs=-1          # 병렬에 쓸 cpu 코어 개수 / -1 : 전체 cpu
)

#모델 학습
rf_model.fit(X_train, y_train) # 학습용 데이터 2개

#예측
rf_predict = rf_model.predict(X_test) # 검증용 데이터

In [22]:
#XGBoost
from xgboost import XGBRegressor

#XGBoost 객체 생성
xg_model = XGBRegressor(
    n_estimators=100,     # 트리를 몇 개 만들지 (여기선 "보완 작업을 몇 번 반복할지"에 가까움)
    learning_rate=0.1,    # 이전 트리가 찾은 보완 내용을 얼마나 강하게 반영할지 (낮을수록 조심스럽게 배움)
    random_state=42,      # 재현성 고정
    n_jobs=-1             # 병렬에 쓸 cpu 코어 개수 / -1 : 전체 cpu
)

#모델 학습
xg_model.fit(X_train, y_train)

#예측
xg_predict = xg_model.predict(X_test)

In [23]:
#LightGBM
from lightgbm import LGBMRegressor

#LightGBM 객체 생성
lgb_model = LGBMRegressor(
    n_estimators=100,     # 트리를 몇 개 만들지
    learning_rate=0.1,    # XGBoost와 동일한 개념 (보완 반영 강도)
    min_child_samples=5,
    random_state=42,      # 재현성 고정
    n_jobs=-1             # 병렬에 쓸 cpu 코어 개수 / -1 : 전체 cpu
)

#모델 학습
lgb_model.fit(X_train, y_train)

#예측
lgb_predict = lgb_model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000101 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 836
[LightGBM] [Info] Number of data points in the train set: 731, number of used features: 26
[LightGBM] [Info] Start training from score 2683330702.971272


In [24]:
rf_mae = mean_absolute_error(y_test, rf_predict)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predict))
rf_r2 = r2_score(y_test, rf_predict)
rf_mape = mean_absolute_percentage_error(y_test, rf_predict)
print(f'랜덤포레스트 MAE : {rf_mae}, RMSE = {rf_rmse}, R2 = {rf_r2}, MAPE = {rf_mape}')

xg_mae = mean_absolute_error(y_test, xg_predict)
xg_rmse = np.sqrt(mean_squared_error(y_test, xg_predict))
xg_r2 = r2_score(y_test, xg_predict)
xg_mape = mean_absolute_percentage_error(y_test, xg_predict)
print(f'XGBoost MAE : {xg_mae}, RMSE = {xg_rmse}, R2 = {xg_r2}, MAPE = {xg_mape}')

lgb_mae = mean_absolute_error(y_test, lgb_predict)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_predict))
lgb_r2 = r2_score(y_test, lgb_predict)
lgb_mape = mean_absolute_percentage_error(y_test, lgb_predict)
print(f'LightGBM MAE : {lgb_mae}, RMSE = {lgb_rmse}, R2 = {lgb_r2}, MAPE = {lgb_mape}')

랜덤포레스트 MAE : 114876764.73166668, RMSE = 221494931.16400054, R2 = 0.9929022543090819, MAPE = 0.04112692639665916
XGBoost MAE : 126719440.0, RMSE = 240092967.3695678, R2 = 0.9916602969169617, MAPE = 0.04589947685599327
LightGBM MAE : 123913037.99300885, RMSE = 233102814.2513553, R2 = 0.9921388173981656, MAPE = 0.044059749266272895


In [25]:
# 랜덤포레스트
rf_importance = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("=== 랜덤포레스트 ===")
print(rf_importance.head(10))

# XGBoost
xg_importance = pd.Series(xg_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n=== XGBoost ===")
print(xg_importance.head(10))

# LightGBM
lgb_importance = pd.Series(lgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n=== LightGBM ===")
print(lgb_importance.head(10))

=== 랜덤포레스트 ===
전년동월       0.558662
시도_경기도     0.439191
평균습도(%)    0.000452
월          0.000346
강수량(mm)    0.000305
난방도일       0.000274
냉방도일       0.000159
폭염일수(일)    0.000155
연도         0.000147
시도_전라남도    0.000047
dtype: float64

=== XGBoost ===
전년동월          0.983374
시도_세종특별자치시    0.004330
시도_울산광역시      0.001682
시도_경상북도       0.001613
시도_전라남도       0.001313
시도_충청북도       0.000865
냉방도일          0.000791
평균습도(%)       0.000787
시도_서울특별시      0.000743
난방도일          0.000650
dtype: float32

=== LightGBM ===
전년동월       938
강수량(mm)    392
난방도일       370
평균습도(%)    285
월          232
연도         230
냉방도일       148
폭염일수(일)     97
시도_경기도      53
한파일수(일)     35
dtype: int32


In [ ]:
target = '총전력사용량(kWh)'
weather_cols = ['냉방도일', '난방도일', '폭염일수(일)', '한파일수(일)', '강수량(mm)', '평균습도(%)']

result = {}
for region, g in df.groupby('시도'):                         # 지역별로 데이터를 하나씩 떼어서
    result[region] = g[weather_cols].corrwith(g[target])    # 그 지역 안에서 날씨 6개 vs 사용량 상관계수 계산

corr_table = pd.DataFrame(result).T.round(2)                # 표로 만들고 소수점 2자리로
corr_table.sort_values('냉방도일', ascending=False)  

C:\Users\GRS\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\GRS\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,냉방도일,난방도일,폭염일수(일),한파일수(일),강수량(mm),평균습도(%)
부산광역시,0.81,-0.18,0.69,0.08,0.26,0.25
서울특별시,0.66,-0.13,0.60,0.10,0.42,0.49
제주특별자치도,0.63,0.06,0.63,NaN,-0.08,0.14
인천광역시,0.61,0.08,0.40,0.24,0.31,0.27
광주광역시,0.61,-0.14,0.62,0.11,0.34,0.40
대구광역시,0.59,-0.13,0.62,0.12,0.38,0.30
대전광역시,0.57,-0.11,0.64,0.15,0.34,0.44
세종특별자치시,0.47,-0.10,0.49,0.08,0.24,0.50
경기도,0.44,0.15,0.46,0.24,0.16,0.20
울산광역시,0.29,-0.18,0.15,0.11,0.36,0.24


In [27]:
#저장
# 1. 랜덤포레스트 모델
with open('models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# 2. 학습에 쓴 컬럼 (순서 중요)
with open('models/train_col.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

# 3. 시도 원핫 인코더 (service.py가 읽는 파일)
with open('models/sido_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)